In [ ]:
# ============================================
# LAKEHOUSE PIPELINE — PATH CONFIGURATION
# ============================================
# Set the base_path widget to your Databricks Unity Catalog
# Volume path before running this notebook.
#
# Default: /Volumes/main/lakehouse_pipeline/raw_data
#   Update this default to match your actual catalog name.
#   (Run: SELECT current_catalog() in a SQL notebook to check.)
#
# How to set the value:
#   Interactive  — Edit the widget text box at the top of the notebook.
#   Workflows    — Pass base_path as a job task parameter;
#                  dbutils.widgets picks it up automatically.
#
# See configs/app_config.yaml for full configuration reference.
# ============================================

# Register widget — update the default value to your Volume path
dbutils.widgets.text(
    "base_path",
    "/Volumes/main/lakehouse_pipeline/raw_data",
    "Base Volume Path"
)

BASE_PATH = dbutils.widgets.get("base_path")

print(f"BASE_PATH configured: {BASE_PATH}")

In [ ]:
# ============================================
# SNOWFLAKE CONNECTION CONFIGURATION
# ============================================
# Credentials are read from environment variables ONLY.
# Never put passwords or account URLs directly in notebook cells.
#
# Required environment variables:
#   SNOWFLAKE_ACCOUNT_URL  e.g. xy12345.snowflakecomputing.com
#   SNOWFLAKE_USER         your Snowflake username
#   SNOWFLAKE_PASSWORD     your Snowflake password
#
# Optional (defaults match configs/app_config.yaml):
#   SNOWFLAKE_DATABASE     default: LAKEHOUSE_DB
#   SNOWFLAKE_SCHEMA       default: ANALYTICS
#   SNOWFLAKE_WAREHOUSE    default: LAKEHOUSE_WH
#   SNOWFLAKE_ROLE         default: ACCOUNTADMIN
#
# In Databricks: set via Cluster Environment Variables or Secrets.
# See configs/app_config.yaml for full configuration reference.
# ============================================

import os

snowflakeOptions = {
    "sfURL":       os.environ.get("SNOWFLAKE_ACCOUNT_URL", ""),
    "sfUser":      os.environ.get("SNOWFLAKE_USER", ""),
    "sfPassword":  os.environ.get("SNOWFLAKE_PASSWORD", ""),
    "sfDatabase":  os.environ.get("SNOWFLAKE_DATABASE", "LAKEHOUSE_DB"),
    "sfSchema":    os.environ.get("SNOWFLAKE_SCHEMA",   "ANALYTICS"),
    "sfWarehouse": os.environ.get("SNOWFLAKE_WAREHOUSE","LAKEHOUSE_WH"),
    "sfRole":      os.environ.get("SNOWFLAKE_ROLE",     "ACCOUNTADMIN")
}

# Validate that required secrets are present before proceeding
_missing = [
    k for k, v in snowflakeOptions.items()
    if k in ("sfURL", "sfUser", "sfPassword") and not v
]
if _missing:
    raise EnvironmentError(
        f"Missing required Snowflake environment variables: {_missing}\n"
        "Set SNOWFLAKE_ACCOUNT_URL, SNOWFLAKE_USER, and SNOWFLAKE_PASSWORD "
        "in your environment before running this notebook."
    )

print("Snowflake configuration loaded from environment variables.")
print(f"Connecting to: {snowflakeOptions['sfDatabase']}.{snowflakeOptions['sfSchema']}")

In [0]:
top_customers_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/top_customers"
)

daily_revenue_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/daily_revenue"
)

revenue_by_country_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/revenue_by_country"
)

print("Gold Layer Loaded Successfully!")

Gold Layer Loaded Successfully!


In [0]:
display(top_customers_df.limit(10))

display(daily_revenue_df.limit(10))

display(revenue_by_country_df.limit(10))

name,total_spent
Jessica Smith,154490.46
David White,112474.3
Michael Smith,112452.81
Ashley Jackson,103501.46
Christopher Miller,102951.86
Ashley Richardson,100192.71
Julie Lee,96616.92
Jordan Jones,92498.89
Christopher Davis,92061.75
Michael Davis,89491.83


order_date,daily_revenue
2025-01-01,623581.08
2025-01-02,634099.11
2025-01-03,522342.88
2025-01-04,509208.23
2025-01-05,643519.35
2025-01-06,630601.67
2025-01-07,621982.97
2025-01-08,544573.66
2025-01-09,611879.66
2025-01-10,640430.86


country,total_revenue
USA,2.269067536E7
India,2.21704257E7
Brazil,2.160822529E7
Japan,2.148327079E7
Canada,2.102396981E7
UK,2.088020462E7
Germany,2.057515548E7
Australia,2.050444894E7
Singapore,2.032287347E7
France,2.026508294E7


In [0]:
print("Testing Snowflake Connection...")

test_df = spark.createDataFrame(
    [(1, "test")],
    ["id", "name"]
)

test_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "CONNECTION_TEST") \
    .mode("overwrite") \
    .save()

print("Snowflake Connection Successful!")

Testing Snowflake Connection...
Snowflake Connection Successful!


In [0]:
top_customers_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "TOP_CUSTOMERS") \
    .mode("overwrite") \
    .save()

print("TOP_CUSTOMERS Exported!")

TOP_CUSTOMERS Exported!


In [0]:
daily_revenue_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "DAILY_REVENUE") \
    .mode("overwrite") \
    .save()

print("DAILY_REVENUE Exported!")

DAILY_REVENUE Exported!


In [0]:
revenue_by_country_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "REVENUE_BY_COUNTRY") \
    .mode("overwrite") \
    .save()

print("REVENUE_BY_COUNTRY Exported!")

REVENUE_BY_COUNTRY Exported!


In [0]:
print("Top Customers Count:", top_customers_df.count())

print("Daily Revenue Count:", daily_revenue_df.count())

print("Revenue By Country Count:", revenue_by_country_df.count())

Top Customers Count: 9370
Daily Revenue Count: 366
Revenue By Country Count: 10


In [0]:
print("SNOWFLAKE EXPORT COMPLETED SUCCESSFULLY!")

SNOWFLAKE EXPORT COMPLETED SUCCESSFULLY!
